# 02 — Synthetic Targets & Baseline Model (AutoCostX Free Edition)

This notebook:
1. Loads the processed feature set from `data/processed/vehicle_master_features.parquet`.
2. Creates **synthetic should‑cost targets (P50/P90)** using lightweight DFMA‑inspired rules and a **Wright’s Law** learning curve parameter.
3. Trains a quick **baseline GBM (XGBoost)** to predict P50 cost.
4. Saves the labeled dataset and model artifacts.

> **Note:** Targets here are **synthetic placeholders** so you can demonstrate the full pipeline.
> They are *transparent and swappable* with real BOM/should‑cost data later.

In [ ]:
# Imports and paths
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from xgboost import XGBRegressor
import joblib

# Project directories
REPO_ROOT = Path("..").resolve()
DATA_PROCESSED = REPO_ROOT / "data" / "processed"
MODELS_DIR = REPO_ROOT / "models"

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Input from notebook 01
features_path = DATA_PROCESSED / "vehicle_master_features.parquet"
features_path

In [ ]:
# Load features (created by 01_load_merge_features)
feat = pd.read_parquet(features_path)

print("Feature rows:", len(feat))
feat.head(3)

## Create synthetic should‑cost targets

We’ll compute **P50** and **P90** should‑costs as follows:

- **Base proxy** (transparent, tunable):
  \[
    \text{base} = A \cdot \text{price\_band} + B \cdot \text{engine\_complexity\_score} + C \cdot \log(1 + \text{feature\_density})
  \]
- **Wright’s Law** (experience curve):
  \[
    \text{cost} \propto \left(\frac{N}{N_0}\right)^b,\; b=\frac{\ln(\text{learning rate})}{\ln 2}
  \]
- **P50** = base after Wright’s factor; **P90** = P50 × 1.2 (conservative tail).

All weights and learning parameters are **explicitly defined** so you can adjust them.

In [ ]:
# Wright's Law helpers
def learning_exponent(learning_rate: float) -> float:
    """
    Compute exponent b given a learning rate (e.g., 0.85).
    Each doubling of cumulative volume multiplies cost by learning_rate.
    """
    return np.log(learning_rate) / np.log(2.0)

def apply_wrights_cost(base_cost: float, cum_units: float, ref_units: float, learning_rate: float) -> float:
    b = learning_exponent(learning_rate)
    scale = (max(cum_units, 1.0) / max(ref_units, 1.0)) ** b
    return float(base_cost) * float(scale)

def make_synthetic_costs(df: pd.DataFrame,
                         lr: float = 0.85,
                         ref_units: float = 50_000,
                         program_units: float = 200_000) -> pd.DataFrame:
    """
    Create synthetic should-cost labels P50/P90 using:
      - simple DFMA-inspired base formula (transparent weights)
      - Wright's Law experience curve scaling
    """
    out = df.copy()

    # Ensure numeric for the features we use
    for col in ["engine_complexity_score", "feature_density", "price_band", "design_complexity_index"]:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")

    # Tunable weights for the base proxy
    A, B, C = 600.0, 900.0, 350.0

    price_band = out.get("price_band", pd.Series([1]*len(out))).fillna(1)
    engine_cx = out.get("engine_complexity_score", pd.Series([1.0]*len(out))).fillna(1.0)
    feat_dens = out.get("feature_density", pd.Series([0.0]*len(out))).fillna(0.0)

    base = A * price_band + B * engine_cx + C * np.log1p(feat_dens)

    # Apply Wright's Law to get P50
    p50 = [apply_wrights_cost(bc, program_units, ref_units, lr) for bc in base]
    out["should_cost_p50"] = np.maximum(100.0, np.array(p50))

    # P90 = +20% uplift (adjust as needed)
    out["should_cost_p90"] = out["should_cost_p50"] * 1.20
    return out

# Generate labeled dataset
labeled = make_synthetic_costs(feat, lr=0.85, ref_units=50_000, program_units=200_000)
labeled[["should_cost_p50", "should_cost_p90"]].describe()

In [ ]:
labeled_path = DATA_PROCESSED / "vehicle_master_labeled.parquet"
labeled.to_parquet(labeled_path, index=False)
labeled_path

## Train a baseline GBM (XGBoost)

We’ll predict **P50 should‑cost** from a compact, interpretable feature set:

- `engine_complexity_score`  
- `feature_density`  
- `price_band`  
- `design_complexity_index`  

We’ll report **MAE** and **R²** on a 20% holdout and save the model to `models/`.

In [ ]:
target_col = "should_cost_p50"
feature_cols = ["engine_complexity_score", "feature_density", "price_band", "design_complexity_index"]

# Keep only rows with the needed columns
data = labeled[feature_cols + [target_col]].copy().dropna()
X = data[feature_cols]
y = data[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

model = XGBRegressor(
    n_estimators=600,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42,
    n_jobs=-1,
)
model.fit(X_train, y_train)
pred = model.predict(X_test)

mae = mean_absolute_error(y_test, pred)
r2 = r2_score(y_test, pred)
print(f"MAE: {mae:,.2f}  |  R²: {r2:.3f}")

# Save model
model_path = MODELS_DIR / "baseline_xgb.pkl"
joblib.dump(model, model_path)
model_path

card = '''\
# AutoCostX Baseline Model — Model Card

**Date:** auto-generated

## Objective
Predict synthetic **should‑cost P50** from design/feature proxies.

## Data
- Input: `data/processed/vehicle_master_features.parquet`
- Output (labeled): `data/processed/vehicle_master_labeled.parquet`

## Target Construction (Synthetic)
- Base cost: weighted combination of `price_band`, `engine_complexity_score`, `feature_density`.
- Experience curve (Wright’s Law): cost scales with cumulative production via exponent *b = log(learning_rate)/log(2)*.
- P90: +20% uplift over P50.

## Modeling
- Algorithm: XGBoost Regressor (GBM).
- Features: `engine_complexity_score`, `feature_density`, `price_band`, `design_complexity_index`.
- Metrics: MAE, R² on 20% test split.

## Assumptions & Limitations
- Labels are **synthetic** and for demonstration; replace with proprietary BOM/should‑cost data when available.
- DFMA ideas informed feature engineering; no commercial DFMA tools/data used.

## Repro Steps
1) Run `notebooks/01_load_merge_features.ipynb`.
2) Run this notebook to generate labels, train the model, and save artifacts.
'''
(MODELS_DIR / "model_card.md").write_text(card, encoding="utf-8")
MODELS_DIR / "model_card.md"